## Who Worked On This Part?
Aarya Ray Chaudhuri: training, tuning and hosting the mushroom model on AWS SageMaker.

# Mushroom model on AWS SageMaker
This notebook trains and tunes our model on AWS and puts it online behind an HTTPS url, so the J backend can ask it for predictions.

It only runs inside SageMaker. We used a **notebook instance** with the `LabRole`: setting up SageMaker Studio failed in AWS Academy, because it tries to create new IAM roles and the lab doesn't allow that. Clone the repo in the notebook instance's terminal and open this file from `Mushroom/`. It won't run on a laptop: it needs the AWS role of the SageMaker machine. We keep the outputs in the repo, as the assignment asks for the SageMaker notebook.

The steps:
1. upload the raw csv to S3
2. one training job with the settings from notebook 2 (our baseline on AWS)
3. a hyperparameter tuning job that tries 12 settings
4. host the best model on an endpoint
5. put an HTTPS url with an api key in front of it (API Gateway + Lambda)

All the model code lives in `model/` and is the same code we test locally (`pytest tests` in `Mushroom/`). `MODEL_PLAN.md` explains every choice.

## Setup
Newer SageMaker images come with version 3 of the SageMaker Python SDK, which works very differently. We pin version 2, the one the AWS course uses. After this cell, restart the kernel once (Kernel → Restart).

In [1]:
%pip install -q "sagemaker>=2.257,<3"

Note: you may need to restart the kernel to use updated packages.


We start a SageMaker session and get the role, the default S3 bucket and the region. In AWS Academy the role is the `LabRole`. The data path uses the same ROOT/DATA pattern as our other notebooks.

In [2]:
import os
os.environ['SAGEMAKER_SUPPRESS_V2_WARNING'] = '1'

import json
import time
import zipfile
from pathlib import Path

import boto3
import sagemaker

# walk up from where the notebook runs to the repo root, so the data path works on every machine
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Mushroom' / 'datasets').is_dir())
MUSHROOM = ROOT / 'Mushroom'
DATA = MUSHROOM / 'datasets'

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
prefix = 'skooma-cats/mushroom'
region, bucket

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


('us-east-1', 'sagemaker-us-east-1-349633751165')

## Which container
SageMaker runs our scripts in a ready-made scikit-learn container. We use the newest one, **1.9-0** (scikit-learn 1.9.0, pandas 2.2.3, Python 3.12). The model is trained **and** hosted in that same container, because a pickled sklearn model is only safe to load with the same sklearn version it was made with.

SDK version 2 doesn't know the name 1.9-0 yet, so we build the image address ourselves. The account numbers come from the SageMaker SDK's own list of images.

In [3]:
# account that hosts the sagemaker scikit-learn images in each region
ACCOUNTS = {'us-east-1': '683313688378', 'us-east-2': '257758044811', 'us-west-2': '246618743249', 'eu-west-1': '141502667606', 'eu-central-1': '492215442770'}
# if the training job says the image doesn't exist, use '1.4-2-cpu-py3' (sklearn 1.4.2, our tests pass on it too)
IMAGE_TAG = '1.9-0-cpu-py3'
image = f'{ACCOUNTS[region]}.dkr.ecr.{region}.amazonaws.com/sagemaker-scikit-learn:{IMAGE_TAG}'
image

'683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.9-0-cpu-py3'

## Data to S3
We upload the **raw** csv, not `mushroom_clean.csv`. The clean file already has measurements filled in by an imputer that saw all 5000 rows, including the ones we test on. `train.py` does the cleaning steps itself and keeps the imputer inside the model, so it only learns from the training rows.

In [4]:
train_s3 = session.upload_data(str(DATA / 'mushroom_project_dataset.csv'), bucket=bucket, key_prefix=f'{prefix}/data')
train_s3

's3://sagemaker-us-east-1-349633751165/skooma-cats/mushroom/data/mushroom_project_dataset.csv'

## Baseline training job
First one job with the same random forest as in notebook 2 (200 trees, `min_samples_leaf=3`). `train.py` prints every metric as `name=value`. SageMaker reads them from the log with the regexes in `aws/metric_definitions.json`, and our tests check that those regexes find every metric.

In [5]:
from sagemaker.sklearn import SKLearn

metric_definitions = json.loads((MUSHROOM / 'aws' / 'metric_definitions.json').read_text())
estimator = SKLearn(
    entry_point='train.py', source_dir=str(MUSHROOM / 'model'), image_uri=image, role=role,
    instance_type='ml.m5.large', instance_count=1,
    hyperparameters={'model': 'rf', 'n-estimators': 200, 'min-samples-leaf': 3},
    metric_definitions=metric_definitions, output_path=f's3://{bucket}/{prefix}/models',
    base_job_name='mushroom-rf', sagemaker_session=session,
)

This starts the job and waits for it. It takes a few minutes, most of it starting the machine.

In [6]:
estimator.fit({'train': train_s3})

INFO:sagemaker:Creating training-job with name: mushroom-rf-2026-10-10-21-17-42-186


2026-10-10 21:17:43 Starting - Starting the training job...
2026-10-10 21:17:59 Starting - Preparing the instances for training...
2026-10-10 21:18:27 Downloading - Downloading input data...
2026-10-10 21:19:17 Training - Training image download completed. Training in progress../opt/venv/lib64/python3.12/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-10-10 21:19:21,483 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-10-10 21:19:21,484 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-10 21:19:21,485 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-10-10 21:19:21,505 sagemaker_sklearn_contain

The final value of every metric, straight from the job description.

In [7]:
job = session.sagemaker_client.describe_training_job(TrainingJobName=estimator.latest_training_job.name)
baseline = {m['MetricName']: m['Value'] for m in job['FinalMetricDataList']}
baseline, job['BillableTimeInSeconds']

({'cv_auc_std': 0.018300000578165054,
  'test_auc': 0.8345999717712402,
  'test_precision': 0.5238999724388123,
  'test_accuracy': 0.6520000100135803,
  'test_recall': 0.8873000144958496,
  'val_precision': 0.4952000081539154,
  'threshold': 0.2492000013589859,
  'test_f1': 0.6588000059127808,
  'val_auc': 0.8363000154495239,
  'val_recall': 0.9014000296592712,
  'train_auc': 0.973800003528595,
  'cv_auc': 0.8269000053405762,
  'test_pr_auc': 0.7835999727249146},
 99)

The baseline job gets **val AUC 0.836**. Retrained on train + validation it gets **test AUC 0.835**. At the threshold 0.249 (picked on validation for 90% recall) it catches 88.7% of the poisonous mushrooms in the test part, with precision 0.524 and accuracy 0.652. The recall is just under the 90% we aimed for: validation has only 284 poisonous mushrooms, so a threshold picked there moves a little on new rows.

These are exactly the numbers `python model/train.py` gives on our laptop (val AUC 0.8363, test AUC 0.8346, threshold 0.2492), even though the container has scikit-learn 1.9.0 and the laptop 1.8. Same code, same split and same `random_state`, so the AWS job really is the model we tested locally. The job was billed for 99 seconds of `ml.m5.large`.

## Hyperparameter tuning job
SageMaker trains 12 forests with different settings, 2 at a time, and uses what it learned from earlier jobs to pick the next settings (Bayesian search). What we tune:
- `n-estimators` (100 to 600): more trees average away more noise, but cost time
- `min-samples-leaf` (1 to 15): bigger leaves can't memorise single rows, which matters with flipped labels
- `max-features` (0.1 to 0.8): fewer features per split makes the trees more different from each other

We use the same 70/15/15 split as `3MushroomPredict.ipynb` (`split()` in `model/mushroom_model.py`, a test checks that the rows are the same). Every job trains on the 70% train part and is scored on the 15% validation part: the objective is **val_auc**. The threshold is picked on validation too. Only then the chosen forest is retrained on train + validation and scored once on the 15% test part. The test part is never used to choose, otherwise we would be tuning on the test set (snoop bias).

In [8]:
from sagemaker.tuner import ContinuousParameter, HyperparameterTuner, IntegerParameter

tuner = HyperparameterTuner(
    estimator, objective_metric_name='val_auc', objective_type='Maximize', metric_definitions=metric_definitions,
    hyperparameter_ranges={
        'n-estimators': IntegerParameter(100, 600),
        'min-samples-leaf': IntegerParameter(1, 15),
        'max-features': ContinuousParameter(0.1, 0.8),
    },
    max_jobs=12, max_parallel_jobs=2, base_tuning_job_name='mushroom-rf-tune',
)

Start the tuning job and wait (around 20 to 30 minutes).

In [9]:
tuner.fit({'train': train_s3})

INFO:sagemaker:Creating hyperparameter tuning job with name: mushroom-rf-tune-261010-2120


...........................................................................................................................................................................................................!


Every try, best first. We save the table in `results/` so notebook 4 can compare it with the local models.

In [10]:
trials = tuner.analytics().dataframe().sort_values('FinalObjectiveValue', ascending=False)
(MUSHROOM / 'results').mkdir(exist_ok=True)
trials.to_csv(MUSHROOM / 'results' / 'sagemaker_tuning.csv', index=False)
trials[['n-estimators', 'min-samples-leaf', 'max-features', 'FinalObjectiveValue', 'TrainingElapsedTimeSeconds']]

,n-estimators,min-samples-leaf,max-features,FinalObjectiveValue,TrainingElapsedTimeSeconds
2,593.0,1.0,0.798656,0.8500,179.0
1,500.0,2.0,0.183357,0.8498,94.0
3,543.0,2.0,0.727096,0.8495,154.0
4,600.0,3.0,0.710889,0.8472,159.0
0,512.0,2.0,0.744999,0.8471,149.0
6,600.0,1.0,0.537029,0.8468,144.0
9,550.0,4.0,0.601249,0.8458,134.0
5,597.0,4.0,0.473640,0.8454,129.0
10,454.0,4.0,0.343142,0.8434,129.0
8,206.0,8.0,0.209989,0.8300,69.0


The best try uses 593 trees, `min_samples_leaf=1` and `max_features=0.80`: **val AUC 0.850**, against 0.836 for the baseline (+0.014). What the 12 tries show:
- **Leaf size is what matters.** Every try with leaves of 1 to 4 rows scores between 0.843 and 0.850. The three tries with big leaves (8, 12 and 14 rows) are the worst (0.817 to 0.830): big leaves hold the forest back too much here.
- **`max_features` hardly matters.** The second best try uses 0.18 instead of 0.80 and scores the same (0.8498 vs 0.8500).
- All 12 jobs together took 26 minutes of machine time, about 17 minutes of waiting with 2 at a time.

Is +0.014 a real gain? The baseline's AUC moves 0.018 between folds (`cv_auc_std`), so the gain is smaller than the noise. One validation set of 750 rows can't tell these forests apart; the next cell also looks at the cross-validation score.

## The best job
All metrics of the winning job, including the test numbers. We look at the test numbers only now, for the model we already picked. They are saved together with the place of the model file in S3.

In [11]:
best_job = tuner.best_training_job()
best = session.sagemaker_client.describe_training_job(TrainingJobName=best_job)
best_result = {'job': best_job, 'hyperparameters': best['HyperParameters'],
               'metrics': {m['MetricName']: m['Value'] for m in best['FinalMetricDataList']},
               'model_data': best['ModelArtifacts']['S3ModelArtifacts']}
(MUSHROOM / 'results' / 'sagemaker_best.json').write_text(json.dumps(best_result, indent=2))
best_result['metrics']

{'ObjectiveMetric': 0.8500000238418579,
 'cv_auc_std': 0.02199999988079071,
 'test_auc': 0.8421000242233276,
 'test_precision': 0.5048999786376953,
 'test_accuracy': 0.628000020980835,
 'test_recall': 0.9014000296592712,
 'val_precision': 0.5038999915122986,
 'threshold': 0.18709999322891235,
 'test_f1': 0.6473000049591064,
 'val_auc': 0.8500000238418579,
 'val_recall': 0.9014000296592712,
 'train_auc': 1.0,
 'cv_auc': 0.8263000249862671,
 'test_pr_auc': 0.7817000150680542}

On cross-validation of the train part the tuned forest scores **0.826 ± 0.022**, the same as the baseline (0.827 ± 0.018). So its better validation AUC probably comes from these particular 750 rows, not from a clearly better forest.

On test it is a little better: AUC 0.842 vs 0.835, and recall 0.901 vs 0.887, so it does reach the 90% we aimed for. Its threshold is lower (0.187 vs 0.249), so it calls more mushrooms poisonous: precision drops to 0.505 (from 0.524) and accuracy to 0.628 (from 0.652).

Train AUC is 1.000: with leaves of 1 row every tree memorises its training rows. The test score still holds up, because averaging 593 different trees cancels most of that out (bagging).

We host this tuned forest, because the tuning job picked it on validation without looking at test. Notebook 4 compares it with the other models on the same test part.

## Host the model
The endpoint loads `model.tar.gz` and answers with `model/inference.py`. We host it on a normal endpoint on `ml.c5.large` (2 vCPU, 4 GB). It is always warm, but it is paid per hour while it runs (about $0.10), so we turn it off when nobody needs it (last cell).

**Path that was blocked: `ml.t2.medium`**, the cheapest machine. The AWS Academy lab policy denies it (`AccessDeniedException ... with an explicit deny`), and we aren't allowed to read that policy. So we tested which machine types the lab allows: we created an endpoint config for each type and deleted it straight away, which is free because no machine starts. Allowed: `ml.m5.large`, `ml.m5.xlarge`, `ml.c5.large`, `ml.c5.xlarge`. Blocked: `ml.t2.medium`, `ml.t2.large`, `ml.m4.xlarge`, `ml.m6i.large`, `ml.c6i.large`. `ml.c5.large` is the cheapest allowed one, and its 4 GB is plenty: the biggest forest the tuning job can pick takes about 230 MB.

**Path we tried and dropped: serverless inference.** It only costs money when a request comes in, which would suit a website with a few requests a day. On 10 Oct the serverless endpoint crashed at startup. The CloudWatch log said `PermissionError: [Errno 13] Permission denied: '/etc/sagemaker-nginx.conf'`: the scikit-learn container writes its web server config to `/etc`, and a serverless endpoint runs the container on a read-only file system. That path is fixed inside AWS's container, so this container can't run serverless. Our own code never got to load. We keep the switch (`SERVERLESS`) to show what we tried.

The endpoint always gets the same name, so the url of the API never changes. This cell only needs the setup cells above it and `results/sagemaker_best.json`, so it is also the "turn it back on" cell. If the endpoint is already running, it leaves it alone. Otherwise it first removes anything a failed or turned-off attempt left behind under the same name (AWS refuses to reuse a name that is still taken), then starts the endpoint. To put a new model online, turn the endpoint off first (last cell).

In [ ]:
from botocore.exceptions import ClientError
from sagemaker.serverless import ServerlessInferenceConfig
from sagemaker.sklearn import SKLearnModel

ENDPOINT = 'skooma-mushroom'
# serverless crashes with this container (see above), so it stays off
SERVERLESS = False
sm = session.sagemaker_client


def endpoint_status():
    found = sm.list_endpoints(NameContains=ENDPOINT)['Endpoints']
    return next((e['EndpointStatus'] for e in found if e['EndpointName'] == ENDPOINT), None)


# an endpoint that is still starting or stopping from an earlier run has to finish first
while endpoint_status() in ('Creating', 'Updating', 'Deleting'):
    time.sleep(30)

if endpoint_status() == 'InService':
    print('the endpoint is already running')
else:
    # a failed or turned-off attempt can leave the endpoint or its config behind, then the name is taken
    for delete in [lambda: sm.delete_endpoint(EndpointName=ENDPOINT), lambda: sm.delete_endpoint_config(EndpointConfigName=ENDPOINT)]:
        try:
            delete()
        except ClientError:
            pass
    while endpoint_status() is not None:
        time.sleep(10)
    best_result = json.loads((MUSHROOM / 'results' / 'sagemaker_best.json').read_text())
    model = SKLearnModel(model_data=best_result['model_data'], role=role, entry_point='inference.py',
                         source_dir=str(MUSHROOM / 'model'), image_uri=image, sagemaker_session=session)
    if SERVERLESS:
        model.deploy(endpoint_name=ENDPOINT, serverless_inference_config=ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=2))
    else:
        model.deploy(endpoint_name=ENDPOINT, initial_instance_count=1, instance_type='ml.c5.large')

We send the four example requests from `aws/examples/` straight to the endpoint. `full`, `from_form` and `batch` should get a prediction, `invalid` should get an error message.

In [ ]:
runtime = boto3.client('sagemaker-runtime', region_name=region)
for name in ['full', 'from_form', 'batch', 'invalid']:
    body = (MUSHROOM / 'aws' / 'examples' / f'{name}.json').read_text()
    answer = runtime.invoke_endpoint(EndpointName=ENDPOINT, ContentType='application/json', Body=body)
    print(name, answer['Body'].read().decode())

## A public HTTPS url for the backend
The endpoint itself only accepts requests signed with AWS keys, and AWS Academy keys expire after a few hours. So we put API Gateway in front of it: a public HTTPS url that checks an api key and passes the request to a small Lambda (`aws/lambda_function.py`), which calls the endpoint. Everything is described in `aws/gateway.yaml` (CloudFormation), so the whole thing can be rebuilt with this one cell.

First the Lambda code goes to S3, under a new name every time so CloudFormation notices a change.

In [ ]:
code_key = f'{prefix}/lambda/lambda-{int(time.time())}.zip'
with zipfile.ZipFile('/tmp/lambda.zip', 'w') as z:
    z.write(MUSHROOM / 'aws' / 'lambda_function.py', 'lambda_function.py')
boto3.client('s3', region_name=region).upload_file('/tmp/lambda.zip', bucket, code_key)
code_key

Then we create the stack, or update it if it already exists, and publish the routes to the `prod` stage. The Lambda runs with the same role as this notebook. If CloudFormation says Lambda can't use that role, put the LabRole arn here instead (IAM → Roles → LabRole).

In [ ]:
STACK = 'skooma-mushroom-api'
LAMBDA_ROLE = role
cfn = boto3.client('cloudformation', region_name=region)
params = [{'ParameterKey': k, 'ParameterValue': v} for k, v in
          {'EndpointName': ENDPOINT, 'LambdaRoleArn': LAMBDA_ROLE, 'CodeBucket': bucket, 'CodeKey': code_key}.items()]
template = (MUSHROOM / 'aws' / 'gateway.yaml').read_text()

existing = [s['StackName'] for s in cfn.list_stacks(StackStatusFilter=['CREATE_COMPLETE', 'UPDATE_COMPLETE', 'UPDATE_ROLLBACK_COMPLETE'])['StackSummaries']]
if STACK in existing:
    cfn.update_stack(StackName=STACK, TemplateBody=template, Parameters=params)
    cfn.get_waiter('stack_update_complete').wait(StackName=STACK)
else:
    cfn.create_stack(StackName=STACK, TemplateBody=template, Parameters=params)
    cfn.get_waiter('stack_create_complete').wait(StackName=STACK)

outputs = {o['OutputKey']: o['OutputValue'] for o in cfn.describe_stacks(StackName=STACK)['Stacks'][0]['Outputs']}
# cloudformation doesn't redeploy the api on an update, so we publish the current routes to the prod stage ourselves
boto3.client('apigateway', region_name=region).create_deployment(restApiId=outputs['ApiId'], stageName='prod')
outputs['PredictUrl']

The api key is a secret, so we don't print it (this notebook's output goes into git). We write it to a file in the home folder of the SageMaker machine, outside the repo, and share it with the backend team in private.

In [ ]:
api_key = boto3.client('apigateway', region_name=region).get_api_key(apiKey=outputs['ApiKeyId'], includeValue=True)['value']
(Path.home() / 'mushroom_api_key.txt').write_text(api_key)
print('api key saved, ends with', api_key[-4:])

## Calling it like the backend does
The same HTTPS request the J backend sends with curl: POST the mushroom as JSON, with the key in the `x-api-key` header.

In [ ]:
import urllib.request

body = (MUSHROOM / 'aws' / 'examples' / 'full.json').read_bytes()
request = urllib.request.Request(outputs['PredictUrl'], data=body, headers={'Content-Type': 'application/json', 'x-api-key': api_key})
with urllib.request.urlopen(request, timeout=30) as r:
    print(r.status, r.read().decode())

## Turning it off and on again
The api stack costs nothing while nobody calls it. The endpoint costs money every hour it runs (`ml.c5.large`, about $0.10, so about $2.40 a day), so we delete it when we don't need it. The url and the key stay the same. To turn it back on: run the setup cells, then the "Host the model" cell.

In [ ]:
# set to True to delete the endpoint, the api stack stays so the url and key don't change
TURN_OFF = False
if TURN_OFF:
    sm = boto3.client('sagemaker', region_name=region)
    sm.delete_endpoint(EndpointName='skooma-mushroom')
    sm.delete_endpoint_config(EndpointConfigName='skooma-mushroom')

# Story
_To write after the run, with the real numbers: what the tuning job changed compared with the baseline job, whether that gain is bigger than the noise between folds, how long and how much it cost, and how the AWS model compares with the local models in notebook 4._